# Klint-32M v2: Institutional Financial Fine-Tuning Engine
### Upgrading Causal Autoregressive Foundation Models on 100 Open-Source Assets with PnL-Weighted Loss
**Author**: Akhilesh Varma ([@akhverm](https://huggingface.co/akhverm)) | **Repository**: [ak495867/Klint-32M](https://github.com/ak495867/Klint-32M)

---

### Abstract & Financial Motivation
Standard foundation time-series models optimize symmetrical **Cross-Entropy** loss:
$$\mathcal{L}_{\text{CE}} = -\sum_{i} y_i \log \hat{y}_i$$

In financial markets, this induces severe trading pathology:
1. **Symmetric Loss Blindness**: A 2-pip error during a quiet sideways market is penalized the same as a 200-pip error during a flash crash.
2. **Directional Indifference**: Predicting $+0.05\%$ when the market moves $-0.05\%$ yields identical cross-entropy to predicting $+0.15\%$ when the market moves $+0.05\%$, yet one bankrupts a momentum portfolio while the other captures positive alpha.
3. **Single-Asset Overfitting**: Pre-training exclusively on one asset (e.g. SOL 1.59M bars) leads to market regime blindness when exposed to cross-market shifts in equities, commodities, or fixed income.

This notebook fine-tunes `klint_32m_release.pt` into **Klint-32M v2** using:
* **100 Liquid Institutional Open-Source Assets**: 40 US Equities, 20 ETFs, 15 Crypto Macro, 10 Commodities, 10 Rates/Bonds, and 5 FX Pairs.
* **PnL-Weighted Loss**: Scales loss dynamically by realized volatility magnitude:
  $$w_t = 1.0 + \lambda_{\text{pnl}} \cdot |r_t^{\text{body}}| \cdot 100$$
* **Asymmetric Directional Hinge Penalty**: Directly penalizes wrong-sign directional forecasts:
  $$\mathcal{L}_{\text{dir}} = \operatorname{ReLU}(-\hat{r}_{\text{pred}} \cdot r_{\text{realized}}) \cdot 100$$
* **Zero Google Drive Dependency**: Operates entirely in `./checkpoints/` with automated Hugging Face acquisition.


In [ ]:
# Stage 1: Environment Setup & Hardware Verification
import os
import sys
import torch

print(f"PyTorch Version : {torch.__version__}")
gpu_available = torch.cuda.is_available()
print(f"CUDA Available  : {gpu_available}")

if gpu_available:
    device = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Hardware    : {gpu_name} ({vram_gb:.2f} GB VRAM)")
else:
    device = "cpu"
    print("Warning: Running on CPU. For fast training, enable a GPU runtime (Runtime > Change runtime type > T4 GPU).")

# Install dependencies if running in Colab
try:
    import google.colab
    IN_COLAB = True
    print("\n[Colab Detected] Syncing repository & installing dependencies...")
    !pip install -q yfinance huggingface_hub matplotlib seaborn
    if os.path.exists("Klint-32M"):
        %cd Klint-32M
        !git pull origin main
    elif not os.path.exists("src/klint"):
        !git clone -q https://github.com/ak495867/Klint-32M.git
        %cd Klint-32M
    sys.path.insert(0, os.path.abspath("src"))
    !pip install -q -e .
except ImportError:
    IN_COLAB = False
    sys.path.insert(0, os.path.abspath("src"))
    print("\n[Local / Dedicated Server] Using local environment.")

print("Environment setup complete.")


---
## Stage 2: Foundation Weights Acquisition (Warm-Start)
We load the pre-trained **Klint-32M** foundation weights ($28,642,560$ parameters).
* If `checkpoints/klint_32m_release.pt` is not present locally, it is automatically fetched directly from Hugging Face (`akhverm/Klint-32M`).
* Standalone model checkpoints (`checkpoints/klint_32m_best.pt` and `checkpoints/tokenizer_best.pt`) are automatically bundled if available.


In [ ]:
# Stage 2: Load or Download Foundation Checkpoints
from klint.eval.bundle_loader import get_or_create_release_bundle
from klint.models.klint_32m import Klint32M, KlintConfig
from klint.tokenizer.factor_tokenizer import FactorTokenizer
from klint.tokenizer.geometric_decoder import GeometricDecoder

os.makedirs("checkpoints", exist_ok=True)
bundle_path = "checkpoints/klint_32m_release.pt"

print("Loading Klint-32M Foundation Model & RVQ Factor Tokenizer...")
model, tokenizer, decoder, cfg, meta = get_or_create_release_bundle(
    bundle_path=bundle_path,
    model_checkpoint="checkpoints/klint_32m_best.pt",
    tokenizer_checkpoint="checkpoints/tokenizer_best.pt",
    hf_repo="akhverm/Klint-32M",
    device=device,
)

param_count = model.count_parameters()
print(f"\nModel Architecture Loaded Successfully:")
print(f"  * Total Trainable Parameters : {param_count:,}")
print(f"  * Hidden Dimension (d_model) : {cfg.d_model}")
print(f"  * Transformer Layers         : {cfg.n_layers}")
print(f"  * Attention Heads            : {cfg.n_heads}")
print(f"  * Price Vocab Size           : {cfg.price_vocab_size}")
print(f"  * Source Training Meta       : {meta}")


---
## Stage 3: Multi-Asset Cross-Market Ingestion Engine (100 Assets)
To eliminate single-asset bias and teach the model universal market dynamics, we train Klint across **100 premier liquid open-source assets** via Yahoo Finance across 6 major asset classes:
1. **US Equities (40 assets)**: Tech & Growth (`AAPL`, `MSFT`, `NVDA`, `GOOGL`, `AMZN`, `META`, `TSLA`, `AVGO`, `AMD`, `QCOM`, `CRM`, `ORCL`, `ADBE`, `PLTR`, `CSCO`), Financials (`JPM`, `V`, `MA`, `BAC`, `WFC`, `MS`, `GS`, `BLK`, `COIN`), Healthcare (`LLY`, `UNH`, `JNJ`, `ABBV`, `MRK`, `PFE`), Consumer (`HD`, `COST`, `WMT`, `PG`, `KO`), Industrials/Energy (`CAT`, `GE`, `XOM`, `CVX`).
2. **Global & Sector ETFs (20 assets)**: Broad Market (`SPY`, `QQQ`, `IWM`, `DIA`, `VOO`, `VTI`), Sector SPDRs (`XLK`, `XLF`, `XLV`, `XLE`, `XLI`, `XLY`, `XLP`, `XLU`, `XLB`), Thematic/Global (`SMH`, `SOXX`, `ARKK`, `EEM`, `INDA`).
3. **Crypto Macro (15 assets)**: `BTC-USD`, `ETH-USD`, `SOL-USD`, `BNB-USD`, `XRP-USD`, `ADA-USD`, `DOGE-USD`, `AVAX-USD`, `LINK-USD`, `DOT-USD`, `NEAR-USD`, `ATOM-USD`, `LTC-USD`, `BCH-USD`, `XLM-USD`.
4. **Commodities (10 assets)**: Gold (`GLD`), Silver (`SLV`), Crude Oil (`USO`), Natural Gas (`UNG`), Broad Commodities (`DBC`), Copper (`CPER`), Platinum (`PPLT`), Wheat (`WEAT`), Corn (`CORN`), Soybeans (`SOYB`).
5. **Rates & Fixed Income (10 assets)**: `TLT` (20+ Yr Treasury), `IEF` (7-10 Yr), `SHY` (1-3 Yr), `BND` (Total Bond), `AGG`, `LQD` (Inv Grade Corp), `HYG` (High Yield), `JNK`, `TIP` (TIPS), `BIL` (1-3 Mo T-Bills).
6. **Foreign Exchange / Global Macro (5 assets)**: `EURUSD=X`, `GBPUSD=X`, `USDJPY=X`, `AUDUSD=X`, `USDCAD=X`.

All series are processed in parallel with multi-threaded downloads, cached to disk, checked for financial invariants ($High \ge \max(Open, Close)$, $Low \le \min(Open, Close)$), decomposed into stationary factors, and quantized into discrete RVQ sequences.


In [ ]:
# Stage 3: Ingest 100 Multi-Asset Market Data & Build Dataloaders
from klint.finetune.multi_asset_dataset import (
    MultiAssetFineTuneDataset,
    build_finetune_dataloaders,
    INSTITUTIONAL_100_TICKERS,
    CORE_12_TICKERS,
    DEFAULT_FINETUNE_TICKERS,
)

# Choose universe: INSTITUTIONAL_100_TICKERS (100 assets) or CORE_12_TICKERS (12 assets)
SELECTED_TICKERS = INSTITUTIONAL_100_TICKERS

print(f"Selected Universe: {len(SELECTED_TICKERS)} liquid institutional assets across 6 classes.")
print(f"Sample Tickers: {SELECTED_TICKERS[:10]} ... + {len(SELECTED_TICKERS)-10} more.")

# Check for local high-resolution files (e.g. data/SOL.npy)
local_files = [f for f in ["data/SOL.npy"] if os.path.exists(f)]
if local_files:
    print(f"Found local baseline dataset: {local_files}")

# Build training and validation dataloaders
# Context window: 256 bars = 768 factor tokens (Price, Range, Activity)
# Max workers: 8 for high-speed parallel fetching with disk caching
print("\nFetching historical OHLCV data & quantizing into RVQ factor sequences...")
train_loader, val_loader = build_finetune_dataloaders(
    tickers=SELECTED_TICKERS,
    local_files=local_files,
    tokenizer=tokenizer,
    batch_size=16 if device == 'cuda' else 4,
    context_bars=256,
    stride_bars=64,
    period="2y",
    interval="1h",
    val_ratio=0.15,
    max_workers=8,
)

print(f"\nDataLoader Summary:")
print(f"  * Training Batches   : {len(train_loader)}")
print(f"  * Validation Batches : {len(val_loader)}")


---
## Stage 4: Financial Utility & Directional Hinge Loss
We configure the custom financial loss function:
$$\mathcal{L}_{\text{total}} = \frac{\mathcal{L}_{\text{price}} + \mathcal{L}_{\text{range}} + \mathcal{L}_{\text{activity}}}{3}$$

### 1. PnL-Weighted Cross-Entropy
$$\mathcal{L}_{\text{weighted}} = \frac{1}{N}\sum_{t=1}^N \left(1.0 + \lambda_{\text{pnl}} \cdot \min(|r_t^{\text{body}}| \cdot 100, 10.0)\right) \cdot \ell_{\text{CE}}(t)$$

### 2. Asymmetric Directional Penalty (Hinge Loss)
$$\hat{r}_{\text{pred}} = \sum_{c=0}^{511} P(c \mid \mathbf{x}) \cdot r_{\text{body}}(c)$$
$$\mathcal{L}_{\text{dir}} = \frac{1}{N}\sum_{t=1}^N \operatorname{ReLU}\left(-\hat{r}_{\text{pred}}(t) \cdot r_{\text{realized}}(t)\right) \cdot 100$$
$$\mathcal{L}_{\text{price}} = \mathcal{L}_{\text{weighted}} + \gamma_{\text{dir}} \cdot \mathcal{L}_{\text{dir}}$$


In [ ]:
# Stage 4: Initialize PnL-Weighted Loss Function
from klint.finetune.loss import PnLWeightedCrossEntropyLoss

# lambda_pnl=2.0 puts 3x weight on 2% moves compared to flat bars
# gamma_dir=1.0 heavily penalizes opposite-sign predictions
criterion = PnLWeightedCrossEntropyLoss(
    tokenizer=tokenizer,
    lambda_pnl=2.0,
    gamma_dir=1.0,
    max_weight_clip=10.0,
).to(device)

print(f"PnL Loss Engine Initialized:")
print(f"  * Price Codebook Tokens Cached : {len(criterion.code_returns)}")
print(f"  * PnL Lambda (Volatility Scale): {criterion.lambda_pnl}")
print(f"  * Directional Gamma (Sign Hinge): {criterion.gamma_dir}")
print(f"  * Sample Body Return Min/Max   : [{criterion.code_returns.min().item():.4f}, {criterion.code_returns.max().item():.4f}]")


---
## Stage 5: Baseline Zero-Shot Evaluation
Before executing fine-tuning, we measure the base model's performance on the 100-asset multi-asset validation dataset to establish an empirical benchmark.


In [ ]:
# Stage 5: Evaluate Baseline Performance Prior to Fine-Tuning
from klint.finetune.pnl_trainer import KlintPnLTrainer

baseline_trainer = KlintPnLTrainer(
    model=model,
    tokenizer=tokenizer,
    criterion=criterion,
    device=device,
)

print("Evaluating Baseline Foundation Model across 100-asset validation split...")
baseline_metrics = baseline_trainer.evaluate(val_loader, max_batches=30)

print("\n" + "=" * 55)
print("  BASELINE (PRE-TRAINED) VALIDATION RESULTS")
print("=" * 55)
print(f"  * Total Loss          : {baseline_metrics['val_loss_total']:.4f}")
print(f"  * Price Factor Loss   : {baseline_metrics['val_loss_price']:.4f}")
print(f"  * Directional Penalty : {baseline_metrics['val_loss_dir']:.4f}")
print(f"  * Directional Hit Rate: {baseline_metrics['val_hit_rate']*100:.2f}%")
print("=" * 55)


---
## Stage 6: Fine-Tuning Execution (Klint-32M -> Klint-32M v2)
We execute warm-start optimization using:
* **Optimizer**: AdamW ($\beta_1 = 0.9, \beta_2 = 0.95$, weight decay $= 0.01$)
* **Learning Rate**: $1 \times 10^{-4}$ decayed to $1 \times 10^{-6}$ via Cosine Annealing
* **Gradient Clipping**: $1.0$ norm
* **Target Output**: `./checkpoints/klint_32m_v2_release.pt`


In [ ]:
# Stage 6: Run Fine-Tuning on 100-Asset Institutional Universe
TOTAL_STEPS = 500       # 500 steps gives rapid convergence; use 1000 for full saturation
EVAL_INTERVAL = 50      # Evaluate and check for best model every 50 steps
OUTPUT_BUNDLE = "checkpoints/klint_32m_v2_release.pt"

trainer = KlintPnLTrainer(
    model=model,
    tokenizer=tokenizer,
    criterion=criterion,
    learning_rate=1e-4,
    min_lr=1e-6,
    total_steps=TOTAL_STEPS,
    grad_clip=1.0,
    device=device,
)

print(f"Starting Fine-Tuning across 100 assets for {TOTAL_STEPS} steps...")
history = trainer.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    steps=TOTAL_STEPS,
    eval_interval=EVAL_INTERVAL,
    save_path=OUTPUT_BUNDLE,
)


---
## Stage 7: Performance Analytics & Candlestick Generation
We inspect the training trajectory and evaluate the upgraded **Klint-32M v2** model by generating future synthetic candlestick trajectories with invariant guarantees:
$$High \ge \max(Open, Close) \quad \text{and} \quad Low \le \min(Open, Close)$$


In [ ]:
# Stage 7.1: Plot Training & Validation Trajectory
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Loss curves
axes[0].plot(history["step"], history["loss"], label="Train PnL Loss", color="#1f77b4", lw=2)
axes[0].plot(history["step"], history["val_loss"], label="Val PnL Loss", color="#ff7f0e", lw=2, marker="o")
axes[0].set_title("Loss Trajectory (PnL-Weighted)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Optimization Step")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Panel 2: Directional Hit Rate Progression
axes[1].plot(history["step"], [h * 100 for h in history["hit_rate"]], label="Train Hit Rate", color="#2ca02c", lw=2)
axes[1].plot(history["step"], [vh * 100 for vh in history["val_hit_rate"]], label="Val Hit Rate", color="#d62728", lw=2, marker="o")
axes[1].axhline(50.0, color="gray", linestyle="--", alpha=0.7, label="Random Guess (50%)")
axes[1].set_title("Directional Hit Rate (%)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Optimization Step")
axes[1].set_ylabel("Accuracy (%)")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# Panel 3: Before vs. After Benchmark
final_val_loss = history["val_loss"][-1] if history["val_loss"] else 0.0
final_val_hr = (history["val_hit_rate"][-1] if history["val_hit_rate"] else 0.0) * 100
base_val_hr = baseline_metrics["val_hit_rate"] * 100

categories = ["Base (Pre-Trained)", "Klint-32M v2 (Fine-Tuned)"]
hr_values = [base_val_hr, final_val_hr]
colors = ["#7f7f7f", "#2ca02c"]

bars = axes[2].bar(categories, hr_values, color=colors, width=0.5)
axes[2].set_title("Validation Hit Rate Comparison", fontsize=13, fontweight="bold")
axes[2].set_ylabel("Directional Hit Rate (%)")
axes[2].set_ylim(40, max(hr_values) + 10)
axes[2].grid(True, alpha=0.3, axis="y")

for bar in bars:
    yval = bar.get_height()
    axes[2].text(bar.get_x() + bar.get_width()/2.0, yval + 0.8, f"{yval:.2f}%", ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# Stage 7.2: Candlestick Forecast Verification with Upgraded Model
# Load the best fine-tuned v2 model
v2_bundle = torch.load(OUTPUT_BUNDLE, map_location=device, weights_only=False)
model.load_state_dict(v2_bundle["model_state_dict"])
model.eval()

print("Loaded fine-tuned Klint-32M v2 bundle.")
print("Generating 15 future market bars conditioned on recent market context...")

# Pick a prompt window from validation set
sample_batch = next(iter(val_loader))
prompt_tokens = sample_batch["inputs"][0:1, :180].to(device)  # First 60 bars

# Autoregressively sample 15 future bars (45 factor tokens)
with torch.no_grad():
    generated_tokens = model.generate_tokens(
        prompt_tokens,
        num_bars=15,
        temperature=0.8,
        top_k=40,
        top_p=0.90,
    )

# De-interleave and decode into continuous factors
p_gen, r_gen, a_gen = FactorTokenizer.deinterleave(generated_tokens)
rec_price, rec_range, rec_activity = tokenizer.decode_tokens(p_gen, r_gen, a_gen)

# Reconstruct exact invariant OHLCV candles
from klint.data.factors import FactorStreams
factor_streams = FactorStreams(
    price_path=rec_price[0].cpu().numpy(),
    range_shape=rec_range[0].cpu().numpy(),
    activity=rec_activity[0].cpu().numpy(),
    anchor_price=100.0,
)
forecast_ohlcv = decoder.decode(factor_streams)

print(f"\nGenerated {len(forecast_ohlcv)} OHLCV Candlesticks:")
print(f"First 5 Forecast Candles [Open, High, Low, Close, Vol]:")
for i in range(min(5, len(forecast_ohlcv))):
    row = forecast_ohlcv[i]
    print(f"  Bar {i+1:2d}: O={row[0]:.2f}, H={row[1]:.2f}, L={row[2]:.2f}, C={row[3]:.2f}, V={row[4]:.0f}")

# Verify 100% invariant enforcement
high_valid = np.all(forecast_ohlcv[:, 1] >= np.maximum(forecast_ohlcv[:, 0], forecast_ohlcv[:, 3]))
low_valid = np.all(forecast_ohlcv[:, 2] <= np.minimum(forecast_ohlcv[:, 0], forecast_ohlcv[:, 3]))
print(f"\nInvariant Validation Check:")
print(f"  * High >= max(Open, Close) : {high_valid} (100% Guaranteed)")
print(f"  * Low <= min(Open, Close)  : {low_valid} (100% Guaranteed)")


---
## Stage 8: Hugging Face Model Hub Push (Optional)
To share your upgraded `klint_32m_v2_release.pt` checkpoint to your Hugging Face model repository ([akhverm/Klint-32M](https://huggingface.co/akhverm/Klint-32M)), run the cell below with your Hugging Face write token.


In [ ]:
# Stage 8: Upload Upgraded v2 Checkpoint to Hugging Face (Optional)
# from huggingface_hub import HfApi, login
#
# HF_TOKEN = "your_hf_write_token_here"  # Or use colab userdata: from google.colab import userdata; userdata.get('HF_TOKEN')
# login(token=HF_TOKEN)
#
# api = HfApi()
# api.upload_file(
#     path_or_fileobj="checkpoints/klint_32m_v2_release.pt",
#     path_in_repo="klint_32m_v2_release.pt",
#     repo_id="akhverm/Klint-32M",
#     repo_type="model",
# )
# print("Successfully uploaded Klint-32M v2 release checkpoint to Hugging Face!")
